### 📊 Experiments Results

🔹 1. Imports

In [17]:
import json
import pandas as pd
from pathlib import Path

🔹 2. Função de interpretação

In [18]:
def interpret(value, metric):
    if metric == "accuracy":
        if value > 0.8: return "🔥 Muito bom"
        elif value > 0.6: return "👍 Ok"
        else: return "⚠️ Baixo"
    
    if metric == "proxy_score":
        if value > 0.75: return "🔥 Forte"
        elif value > 0.5: return "👍 Médio"
        else: return "⚠️ Fraco"

🔹 3. Load + processamento

In [19]:
def load_and_process(path):
    data = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    rows = []

    for item in data:
        if not isinstance(item, dict):
            print("Item estranho:", item)
            continue

        pred = item.get("prediction", "").upper()
        gold = item.get("gold_label", "").upper()

        # 🔥 CORREÇÃO IMPORTANTE AQUI
        steps = item.get("pipeline", {}).get("steps", [])

        evidence_counts = [len(s.get("evidence", [])) for s in steps]

        avg_evidence = sum(evidence_counts) / len(evidence_counts) if evidence_counts else 0

        rows.append({
            "claim_id": item.get("claim_id"),
            "prediction": pred,
            "label": gold,
            "correct": pred == gold,
            "num_steps": len(steps),
            "avg_evidence_per_step": avg_evidence,
        })

    df = pd.DataFrame(rows)

    accuracy = df["correct"].mean()
    evidence_score = df["avg_evidence_per_step"].mean()

    proxy_score = 0.7 * accuracy + 0.3 * min(evidence_score / 5.0, 1.0)

    metrics = {
        "accuracy": accuracy,
        "proxy_score": proxy_score
    }

    return df, metrics

🔹 4. Paths

In [20]:
paths = {
    "baseline": [
        "../outputs/averitec_original_pipeline/gemma3-1b/averitec_dev.jsonl"
    ],
    "iterative": [
        "../outputs/averitec_iterative_pipeline/gemma3-1b/averitec_dev.jsonl"
    ]
}

🔹 5. Carregamento geral

In [21]:
dfs = {}
results = {}

for pipeline, file_list in paths.items():
    dfs[pipeline] = []
    results[pipeline] = []

    for path in file_list:
        df, metrics = load_and_process(path)

        run_name = Path(path).parent.name
        df["run"] = run_name

        dfs[pipeline].append(df)
        results[pipeline].append(metrics)

🔹 6. Concatenar tudo

In [22]:
dfs_concat = {
    k: pd.concat(v, ignore_index=True)
    for k, v in dfs.items()
}

🔹 7. Print por pipeline

In [23]:
from pathlib import Path
from sklearn.metrics import classification_report

import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

def print_pipeline_summary(name, df):
    run_name = Path(path).parent.name
    dataset_name = Path(path).name

    df["run"] = run_name
    df["dataset"] = dataset_name

    for run, df_run in df.groupby("run"):

        accuracy = df_run["correct"].mean()
        evidence_score = df_run["avg_evidence_per_step"].mean()
        

        df_errors = df_run[~df_run["correct"]]

        print("=" * 50)
        print(f"📌 PIPELINE: {name.upper()} - {run}")
        #print(f"📂 ARQUIVO: {run}")
        print(f"📂 DATASET: {df_run['dataset'].iloc[0]}")
        print("\n")
        print('🎯 Accuracy:', round(accuracy, 2), '->', interpret(accuracy, 'accuracy'))
        print("\n")

        print("📊 Total samples:", len(df_run))
        print("✅ Correct predictions:", df_run["correct"].sum())
        print("❌ Errors:", len(df_errors))
        print("=" * 50)
        #print(classification_report(df["label"], df["prediction"]))
        #print("📊 Per-class accuracy:")
        #print(df.groupby("label")["correct"].mean())
#
for name in dfs_concat:
    print_pipeline_summary(name, dfs_concat[name])

📌 PIPELINE: BASELINE - gemma3-1b
📂 DATASET: averitec_dev.jsonl


🎯 Accuracy: 0.57 -> ⚠️ Baixo


📊 Total samples: 500
✅ Correct predictions: 285
❌ Errors: 215
📌 PIPELINE: ITERATIVE - gemma3-1b
📂 DATASET: averitec_dev.jsonl


🎯 Accuracy: 0.58 -> ⚠️ Baixo


📊 Total samples: 288
✅ Correct predictions: 168
❌ Errors: 120


In [24]:

paths = {
    "baseline": [
        "../outputs/averitec_original_pipeline/gemma3-1b/averitec_train.jsonl"
    ],
    "iterative": [
        "../outputs/averitec_iterative_pipeline/gemma3-1b/averitec_train.jsonl"
    ]
}

dfs = {}
results = {}

for pipeline, file_list in paths.items():
    dfs[pipeline] = []
    results[pipeline] = []

    for path in file_list:
        df, metrics = load_and_process(path)

        run_name = Path(path).parent.name
        df["run"] = run_name

        dfs[pipeline].append(df)
        results[pipeline].append(metrics)

dfs_concat = {
    k: pd.concat(v, ignore_index=True)
    for k, v in dfs.items()
}

for name in dfs_concat:
    print_pipeline_summary(name, dfs_concat[name])

FileNotFoundError: [Errno 2] No such file or directory: '../outputs/averitec_iterative_pipeline/gemma3-1b/averitec_train.jsonl'